In [1]:
import requests
import pandas as pd
import io

def fetch_chromatin_modifiers():
    # UniProt REST API URL
    url = "https://rest.uniprot.org/uniprotkb/search"
    
    # Query: Human proteins with GO terms for histone modification activities
    # GO:0016570 (histone modification), GO:0004402 (HAT), GO:0008270 (zinc ion binding - often cofactors)
    query = 'taxonomy_id:9606 AND (go:0016570 OR go:0004402 OR go:0034968)'
    
    params = {
        'query': query,
        'format': 'tsv',
        'fields': 'accession,id,gene_names,cc_catalytic_activity,cc_cofactor',
        'size': 500  # Adjust size as needed
    }

    response = requests.get(url, params=params)
    if response.status_code != 200:
        print(f"Error fetching data: {response.status_code}")
        return None

    # Load into DataFrame
    df = pd.read_csv(io.StringIO(response.text), sep='\t')
    
    # Rename for clarity
    df.columns = ['Accession', 'Entry Name', 'Genes', 'Catalytic Activity', 'Cofactors']
    
    # Minimal cleaning to extract the "Reaction" text
    df['Reaction'] = df['Catalytic Activity'].str.extract(r'REACTION: (.*?);')
    
    return df

# Run the fetcher
data = fetch_chromatin_modifiers()

# Example: Display enzymes that mention specific histone residues (e.g., H3)
h3_modifiers = data[data['Catalytic Activity'].str.contains('H3', na=False, case=False)]
print(h3_modifiers[['Genes', 'Reaction', 'Cofactors']].head(10))

# Export to CSV for your database
# data.to_csv('chromatin_metabolism_db.csv', index=False)

Empty DataFrame
Columns: [Genes, Reaction, Cofactors]
Index: []
